# CNN - CIFAR10 - Tcl Trl

# Import Libraries

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import torch.backends.cudnn as cudnn
import os
import sys
sys.path.insert(0, '..')
from utils import *


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' The Device is set to : {device}')

# Import dataset

- augmented
- normalized
- padded
- shuffled
- CIFAR10

In [17]:
trainloader, testloader, trainset, testset = load_cifar(BATCH_SIZE=32,PATH="./data")

Files already downloaded and verified
Files already downloaded and verified


# CNN Model + Tcl Trl


In [18]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.ad_pool = nn.AdaptiveAvgPool2d(output_size=(6,6))
        
        self.fc1 = nn.Linear(in_features=128 * 6 * 6,out_features= 512, bias= True) 
        self.fc2 = nn.Linear(in_features= 512, out_features=256)
        self.fc3 = nn.Linear(256, 10) 
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.ad_pool(x)

        x = x.view(-1, 128 * 6 * 6)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Model : CNN + Tcl Trl

In [19]:
model = CNN().to(device)

num_params = count_param(model)

print("number of parameters:" , num_params)
print(model)

create model
number of parameters: 3438922


# Model Train and Evaluation

In [20]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= 0.1,
                        momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200)


In [ ]:
def topk(output, target, k):
    correct = 0.0
    batch_size = output.shape[0]
    for sample in range(batch_size):
        topk_sorted = output[sample].sort()[1][:k]
        if target[sample] in topk_sorted:
           correct+=1
        #    print(f'sample {sample} was correct because : {topk_sorted} and {target[sample]}')
    return (correct/batch_size)*100.0
        

In [21]:

def train(epoch):
    file_path = '../results/cifar_10/tcl_trl/CNN_train.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    print('\nEpoch: %d' % epoch)
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    with open(file_path, 'a') as f:
        f.write(f'\nEpoch: {epoch}\n')
        
        for batch_idx, (inputs, targets) in enumerate(trainloader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_summary = f'Train Summary after Epoch: {epoch}, Loss: {train_loss / len(trainloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
        f.write(train_summary)

        print(train_summary)
        model_save_path = f'../results/cifar_10/tcl_trl/epoch_{epoch}_Cnn_train.pth'
        os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
        torch.save(model.state_dict(), model_save_path)
        print(f'Model saved to {model_save_path}')


In [22]:
def test(epoch):
    file_path = '../results/cifar_10/tcl_trl/CNN_test.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        with open(file_path, 'a') as f:
            f.write(f'\nTesting after Epoch: {epoch}\n')
            
            for batch_idx, (inputs, targets) in enumerate(testloader):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                
            test_summary = f'Test Summary after Epoch {epoch}, Loss: {test_loss / len(testloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
            f.write(test_summary)
            
            print(test_summary)

In [23]:
Epoch = 300
for epoch in range(1, Epoch + 1):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0
Epoch: 0, Loss: 1.914, Accuracy: 29.574% (14787/50000)

Loss: 1.728 | Acc: 40.000% (40/100)
Loss: 1.880 | Acc: 36.000% (72/200)
Loss: 1.865 | Acc: 36.000% (108/300)
Loss: 1.870 | Acc: 35.500% (142/400)
Loss: 1.862 | Acc: 35.400% (177/500)
Loss: 1.820 | Acc: 36.667% (220/600)
Loss: 1.820 | Acc: 36.143% (253/700)
Loss: 1.835 | Acc: 35.625% (285/800)
Loss: 1.825 | Acc: 35.556% (320/900)
Loss: 1.824 | Acc: 35.700% (357/1000)
Loss: 1.812 | Acc: 36.364% (400/1100)
Loss: 1.816 | Acc: 35.750% (429/1200)
Loss: 1.814 | Acc: 35.923% (467/1300)
Loss: 1.811 | Acc: 35.500% (497/1400)
Loss: 1.807 | Acc: 35.133% (527/1500)
Loss: 1.811 | Acc: 35.000% (560/1600)
Loss: 1.811 | Acc: 35.588% (605/1700)
Loss: 1.817 | Acc: 35.389% (637/1800)
Loss: 1.816 | Acc: 35.684% (678/1900)
Loss: 1.821 | Acc: 35.400% (708/2000)
Loss: 1.821 | Acc: 35.000% (735/2100)
Loss: 1.814 | Acc: 35.318% (777/2200)
Loss: 1.805 | Acc: 35.609% (819/2300)
Loss: 1.807 | Acc: 35.375% (849/2400)
Loss: 1.803 | Acc: 35.520% (888/2

KeyboardInterrupt: 